# SMS Spam Classifier: Exploratory Data Analysis & Modeling

This notebook covers the end-to-end pipeline of building a robust Spam Email Classifier utilizing Classical Machine Learning.

## 1. Data Loading and Overview

We utilize the Kaggle SMS Spam Collection Dataset. It contains labels (`ham` or `spam`) along with raw text messages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/spam.csv', encoding='latin-1').iloc[:, :2]
df.columns = ['label', 'message']
df.head()

## 2. Exploratory Data Analysis (EDA)
Let's visualize the target variables and identify statistical differences in message composition.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='label', palette='viridis')
plt.title('Class Distribution')
plt.show()

In [ ]:
# Message Length Analysis
df['msg_length'] = df['message'].apply(len)
plt.figure(figsize=(10,6))
sns.histplot(data=df, x='msg_length', hue='label', bins=50, kde=True)
plt.title('Message Length Distribution (Spam vs. Ham)')
plt.xlim(0, 300)
plt.show()

In [ ]:
from wordcloud import WordCloud
spam_words = ' '.join(list(df[df['label'] == 'spam']['message']))
spam_wc = WordCloud(width = 512,height = 512, background_color='black').generate(spam_words)
plt.figure(figsize = (8, 8))
plt.imshow(spam_wc)
plt.axis('off')
plt.title('Spam Message WordCloud')
plt.show()

In [ ]:
ham_words = ' '.join(list(df[df['label'] == 'ham']['message']))
ham_wc = WordCloud(width = 512,height = 512, background_color='white').generate(ham_words)
plt.figure(figsize = (8, 8))
plt.imshow(ham_wc)
plt.axis('off')
plt.title('Ham Message WordCloud')
plt.show()

## 3. Text Preprocessing Walkthrough
The input textual data is subjected to specific preprocessing transformations prior to feature extraction. Examples include lowering strings, escaping punctuation, expanding abbreviations, removing standard stopwords via NLTK, and extracting semantic stems utilizing the `PorterStemmer`.

In [ ]:
import sys
sys.path.append('../src')
from preprocess import clean_text

sample_str = "URGENT! You have won a 1 week FREE membership"
print(f"Raw: {sample_str}")
print(f"Cleaned: {clean_text(sample_str)}")

## 4. Feature Extraction
We use **TF-IDF Vectorization** (Term Frequency - Inverse Document Frequency). This reduces the impact of tokens that occur highly generally across the corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned'] = df['message'].apply(clean_text)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned'])
print(f"Feature matrix extracted with shape: {X.shape}")

## 5. Model Training and Comparison
Utilizing baseline tests utilizing `Multinomial Naive Bayes` against robust models like `Random Forest` and `SVM Linear Kernel`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

y = df['label'].map({'ham': 0, 'spam': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

nb = MultinomialNB()
nb.fit(X_train, y_train)
preds = nb.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(classification_report(y_test, preds))

## 6. Best Model Deep Dive
The SVM or NB models typically resolve with 98% accuracy. Feature Importances via checking log_probabilities can help highlight indicators triggering the model natively.

## 7. Error Analysis
What did the model get wrong? Typically short texts completely devoid of common spam keywords, or long sophisticated emails pretending to be a colleague trigger False Negatives / Positives.

In [ ]:
# Example of checking FN/FP
df_test = pd.DataFrame({'True': y_test, 'Pred': preds})
false_positives = df_test[(df_test['True'] == 0) & (df_test['Pred'] == 1)]
print(f"False Positives detected: {len(false_positives)}")

## 8. Conclusions & Future Work
While Classical ML offers extreme compute efficiency and achieves phenomenal accuracy (>98%), further improvements via sequence capture (LSTM), bidirectional transformers (BERT), or browser context extraction exist for broader deployment capabilities.